Introduction:

Sabrina and I are Microbiology majors, and have served as teaching assistants for MCB3023L; a core Microbiology laboratory class. A key part of the class is the lab practical, which constitutes a large part of the students' grades. It is notorious for being long and difficult.\

We wanted to investigate factors potentially related to performance on the lab practical so that we could help our professor and her future students.

Data Collection:

We started off by creating a survey for MCB3023L asking several questions related to 1) their letter grade performance on the practical and 2) their studying/learning habits related to the lab practical.

We then had to obtain IRB-exemption (since we are collected data off of people) and sponsorship from our professor of whom we TA'd for.

The next part involved contacting three professors who teach MCB3023L and requesting they distribute the survey to their students. 

After data was collected, we began with preprocessing.


A look at the questions asked on the survey: 

How often did you pay close attention during lab sessions?

How many times did you come into the lab outside of class to practice techniques?

How confident were you in your lab techniques before the practical? (rate 1-5 )

How anxious were you about the lab practical? (rate 1-5 )

How many hours per week did you study or review lab materials?

Helpfulness in Studying: Lab Manual (rate 1-5 stars)

Helpfulness in Studying: Quizlet (rate 1-5 stars)

Helpfulness in Studying: Lab Slides (rate 1-5 stars)

Helpfulness in Studying: ELNs (rate 1-5 stars)

Helpfulness in Studying: ChatGPT (rate 1-5 stars)

Do you or have you worked or volunteered in a lab setting outside of MCB3023L? (yes or no)

What grade did you receive on the lab practical? ← (Target!! What we want to predict!! Asked as Letter Grade, not Numerical grade)

🧼 Data Preprocessing
First, we imported the dataset using pandas. Since we don't need the timestamp for our analysis, we dropped that column right away. We also made sure to clean up the column names by stripping any extra whitespace — just in case there were formatting issues from Google Forms.

In [16]:
import pandas as pd

# Load the dataset
file_path = "./MCB3032L (Responses) - Form Responses 1.csv"
df = pd.read_csv(file_path)

# Drop the timestamp column bc we do not need it
df = df.drop(columns=["Timestamp"], errors='ignore')

# Strip leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

# Preview the first few rows
df.head()


,How often did you pay close attention during lab sessions?,"How many times did you come into the lab outside of class to practice techniques? (specifically, the instances were class was dismissed and you stayed to work on techniques, or if you came in at your own times)",How confident were you in your lab techniques before the practical?,How anxious were you about the lab practical?,How many hours per week did you study or review lab materials?,Helpfulness in Studying: Lab Manual,Helpfulness in Studying: Quizlet,Helpfulness in Studying: Lab Slides,Helpfulness in Studying: ELNs,Helpfulness in Studying: ChatGPT,"Do you or have you worked or volunteered in a lab setting outside of MCB3023L, where you use techniques you have learned in MCB3032L?",What grade did you receive on the lab practical?
0,most of the time,0 times,5,3,less than 1 hour,3,1,4,5,1,yes,A (90-100)
1,most of the time,2-3 times,5,2,1-2 hours,1,1,2,1,1,yes,D (60-69)
2,most of the time,2-3 times,4,2,1-2 hours,3,1,5,5,4,yes,B (80-89)
3,most of the time,2-3 times,3,3,less than 1 hour,2,2,5,4,1,yes,B (80-89)
4,most of the time,2-3 times,3,5,less than 1 hour,2,1,3,4,4,yes,B (80-89)


Looking at the data, there is a lot an ML algorithm is not going to get unless we encode it in some way. We have to convert the responses into numerical or Boolean form. 
To prepare our data for modeling, we need to convert some of the survey responses from text into numeric values. This is important because machine learning models don’t work well with raw strings — they need numerical inputs to learn from patterns.

We created a new copy of the dataset and renamed the column containing final lab grades to "Grade" for convenience. Then, we applied ordinal encoding to several of the survey questions that had a natural order (e.g., "never" to "always", or "0 times" to "4+ times").

We also applied binary encoding to the yes/no question about lab experience.

-These mappings are based on all possible answer choices from the original survey — even if not all of them appear in the current data. This ensures our encoding will still work if we collect more responses later.

Before applying the mappings, we checked each column to make sure there were no unexpected answers (misspelled values, blanks, etc.). If any responses didn’t match what we expected, they would show up in the output as “unmapped.”

In [17]:
# Make a fresh copy to encode
df_encoded = df.copy()

# Rename the target column to 'Grade' for convenience
df_encoded.rename(columns={
    "What grade did you receive on the lab practical?": "Grade"
}, inplace=True)

# Define mapping dictionaries for all survey questions (based on original form, not just present values)
ordinal_maps = {
    "How often did you pay close attention during lab sessions?": {
        "never": 1,
        "rarely": 2,
        "sometimes": 3,
        "most of the time": 4,
        "always": 5
    },
    "How many times did you come into the lab outside of class to practice techniques? (specifically, the instances were class was dismissed and you stayed to work on techniques, or if you came in at your own times)": {
        "0 times": 0,
        "1 time": 1,
        "2-3 times": 2,
        "4+ times": 3
    },
    "How many hours per week did you study or review lab materials?": {
        "less than 1 hour": 0,
        "1-2 hours": 1,
        "3-4 hours": 2,
        "5+ hours": 3
    },
    "Do you or have you worked or volunteered in a lab setting outside of MCB3023L, where you use techniques you have learned in MCB3032L?": {
        "no": 0,
        "yes": 1
    }
}

# Optional: Check for any values that aren't mapped (useful when cleaning or adding more data)
print("🔍 Unmapped values (if any):")
for col, mapping in ordinal_maps.items():
    if col in df_encoded.columns:
        print(f"\n{col}")
        for val in df_encoded[col].dropna().unique():
            if val not in mapping:
                print(f"  ❗ Unmapped value: '{val}'")

# Apply the mappings to convert text to numeric values
for col, mapping in ordinal_maps.items():
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].map(mapping)

# Preview the encoded data
df_encoded.head()


🔍 Unmapped values (if any):

How often did you pay close attention during lab sessions?

How many times did you come into the lab outside of class to practice techniques? (specifically, the instances were class was dismissed and you stayed to work on techniques, or if you came in at your own times)

How many hours per week did you study or review lab materials?

Do you or have you worked or volunteered in a lab setting outside of MCB3023L, where you use techniques you have learned in MCB3032L?


,How often did you pay close attention during lab sessions?,"How many times did you come into the lab outside of class to practice techniques? (specifically, the instances were class was dismissed and you stayed to work on techniques, or if you came in at your own times)",How confident were you in your lab techniques before the practical?,How anxious were you about the lab practical?,How many hours per week did you study or review lab materials?,Helpfulness in Studying: Lab Manual,Helpfulness in Studying: Quizlet,Helpfulness in Studying: Lab Slides,Helpfulness in Studying: ELNs,Helpfulness in Studying: ChatGPT,"Do you or have you worked or volunteered in a lab setting outside of MCB3023L, where you use techniques you have learned in MCB3032L?",Grade
0,4,0,5,3,0,3,1,4,5,1,1,A (90-100)
1,4,2,5,2,1,1,1,2,1,1,1,D (60-69)
2,4,2,4,2,1,3,1,5,5,4,1,B (80-89)
3,4,2,3,3,0,2,2,5,4,1,1,B (80-89)
4,4,2,3,5,0,2,1,3,4,4,1,B (80-89)
